# Day 23 — Data pipelines: functions, generators, memory-efficient processing
Objectives:
- Build streaming pipelines with generators.
- Chunk large files.
- Compose transforms with .pipe.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-23`. Read
`python/ds-60day/companion-guides/day23_data_pipelines_generators.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A pipeline is a sequence of stages that turn input into output. Give each
stage one responsibility—read, parse, validate, normalize, filter,
batch, or write—and define the shape it accepts and yields. Small stages
are easier to test and recombine than one function mixing files, rules,
logging, and output.

Generator stages keep memory bounded by yielding one item at a time.
Calling a generator function does not run its body; iteration starts
work and advances state. The stage that opens a resource should own its
`with` block for the entire iteration. Decide whether malformed records
stop the stream, are skipped with evidence, or go to quarantine.

### Vocabulary

- **pipeline:** an ordered composition of input/output stages.
- **stage:** one transformation with a stated input and output contract.
- **streaming:** processing incrementally rather than loading everything.
- **backpressure:** a consumer limiting how quickly upstream work advances.
- **batch:** a bounded group processed or written together.
- **quarantine:** separate retention of invalid records and failure reasons.

## Syntax anatomy

`yield normalized` pauses a generator stage after producing one record.
`yield from iterable` delegates successive values to another iterable.
A consumer such as `for batch in batches(rows, 100):` drives the whole
upstream chain; without consumption, no generator body or file read
occurs.

### Worked example 1 — Compose lazy normalization and filtering

Each stage consumes and yields one record at a time. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
from collections.abc import Iterable, Iterator

def normalize(rows: Iterable[dict[str, str]]) -> Iterator[dict[str, str]]:
    for row in rows:
        yield {"name": row["name"].strip(), "status": row["status"].lower()}

def active_only(rows: Iterable[dict[str, str]]) -> Iterator[dict[str, str]]:
    for row in rows:
        if row["status"] == "active":
            yield row

source = [{"name": " Ada ", "status": "ACTIVE"}, {"name": "Lin", "status": "inactive"}]
list(active_only(normalize(source)))

**Expected observation:** `[{'name': 'Ada', 'status': 'active'}]`. No stage needed all records at once.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Preserve a final partial batch

Batch boundaries are part of the contract. Predict first; then run the next cell.

In [ ]:
def batches(items: Iterable[int], size: int) -> Iterator[tuple[int, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == size:
            yield tuple(batch)
            batch = []
    if batch:
        yield tuple(batch)

list(batches(range(5), 2))

**Expected observation:** `[(0, 1), (2, 3), (4,)]`. The last partial batch is not silently lost.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Test each stage with a tiny list before composing the pipeline.
2. If no work occurs, confirm that a consumer actually iterates the generator.
3. Keep a file's `with` block inside the generator that performs iteration.
4. Count input, accepted, quarantined, and output records to reconcile every path.

**Alternative to compare:** Use eager lists for small data and simple debugging; use lazy stages when memory, streaming, or early stopping matters.

**Boundary to test:** Partial consumption, consumer failure, invalid batch size, final partial batch, malformed first/last rows, and cleanup on exceptions need tests.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
from pathlib import Path
def read_lines(path, chunk=1024):
    with open(path, 'r', encoding='utf-8') as f:
        while True:
            data = f.readlines(chunk)
            if not data: break
            for line in data: yield line.strip()

# Example pipeline
def only_numbers(lines):
    for s in lines:
        if s.isdigit(): yield int(s)

# list(only_numbers(read_lines('data.txt')))


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Build `stream_clean_csv(path)` that opens a UTF-8 CSV inside the generator, yields one normalized row dictionary at a time, and follows a written malformed-row policy. **Constraints:** use `csv.DictReader`, keep the file open during iteration, and do not return a list.
   **Verify:** partially consume two rows, consume the rest, test an empty file, and confirm the handle closes after completion/failure.

2. Compose separate generator stages to filter, map, and batch clean rows. **Contract:** every stage documents input/output shape; batch size must be positive; the final partial batch is yielded.
   **Verify:** on a five-row fixture with size two, assert exact batches, source order, and input = accepted + quarantined + deliberately filtered counts.

### Additional mastery practice

Build one-responsibility lazy stages with explicit resource ownership, error policy, and final-partial-batch behavior.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict when side effects inside a generator body occur: at function call, first iteration, or full materialization.
   **Progressive hint:** Calling a generator function creates an iterator; its body starts on iteration.
   **Verify:** Append an event inside the generator body and assert no event at construction, one at first `next`, and all remaining events only after full consumption.
4. **Tracing:** Consume two items from a five-item generator, then pass it onward. Trace which values the next stage can still see.
   **Progressive hint:** Consumption is stateful and does not rewind automatically.
   **Verify:** Assert the first consumer sees values 0 and 1 and the downstream stage sees exactly 2, 3, and 4—never a restarted sequence.
5. **Implementation:** Implement `batch_rows(rows, size)` yielding tuples and preserving the final partial batch.
   **Progressive hint:** Validate positive size and reset the accumulator after each yield.
   **Verify:** Assert five rows at size two produce `[(0, 1), (2, 3), (4,)]`, empty input produces none, and nonpositive size raises.
6. **Debugging:** Repair a function that returns a generator expression over a file after the surrounding `with` block has already closed the file.
   **Progressive hint:** Own the `with` block inside the generator that performs iteration.
   **Verify:** Partially consume the repaired file generator and then finish it; assert rows remain readable until exhaustion and the handle closes afterward.
7. **Edge case and explanation:** Add a quarantine side channel for malformed rows without making the clean stream eagerly load the entire file.
   **Progressive hint:** Pass a callback/list for bounded error records or yield tagged results.
   **Verify:** Feed valid/invalid/valid rows and assert the clean stream remains lazy, output order is preserved, and accepted plus quarantined counts equal input.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Build `stream_clean_csv(path)` that opens a UTF-8 CSV inside the generator, yields one normalized row dictionary at a time, and follows a written malformed-row policy. **Constraints:** use `csv.DictReader`, keep the file open during iteration, and do not return a list. **Verify:** partially consume two rows, consume the rest, test an empty file, and confirm the handle closes after completion/failure.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Build `stream_clean_csv(path)` that opens a UTF-8 CSV inside the generator, yields one normalized row dictionary at a time, and follows a written malformed-row policy. use `csv....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Compose separate generator stages to filter, map, and batch clean rows. **Contract:** every stage documents input/output shape; batch size must be positive; the final partial batch is yielded. **Verify:** on a five-row fixture with size two, assert exact batches, source order, and input = accepted + quarantined + deliberately filtered counts.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Compose separate generator stages to filter, map, and batch clean rows. every stage documents input/output shape; batch size must be positive; the final partial batch is yielded...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict when side effects inside a generator body occur: at function call, first iteration, or full materialization. **Progressive hint:** Calling a generator function creates an iterator; its body starts on iteration. **Verify:** Append an event inside the generator body and assert no event at construction, one at first `next`, and all remaining events only after full consumption.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict when side effects inside a generator body occur: at function call, first iteration, or full materialization. Calling a generator function creates an iterator; its body s...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Consume two items from a five-item generator, then pass it onward. Trace which values the next stage can still see. **Progressive hint:** Consumption is stateful and does not rewind automatically. **Verify:** Assert the first consumer sees values 0 and 1 and the downstream stage sees exactly 2, 3, and 4—never a restarted sequence.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Consume two items from a five-item generator, then pass it onward. Trace which values the next stage can still see. Consumption is stateful and does not rewind automatically. As...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement `batch_rows(rows, size)` yielding tuples and preserving the final partial batch. **Progressive hint:** Validate positive size and reset the accumulator after each yield. **Verify:** Assert five rows at size two produce `[(0, 1), (2, 3), (4,)]`, empty input produces none, and nonpositive size raises.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement `batch_rows(rows, size)` yielding tuples and preserving the final partial batch. Validate positive size and reset the accumulator after each yield. Assert five rows at...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a function that returns a generator expression over a file after the surrounding `with` block has already closed the file. **Progressive hint:** Own the `with` block inside the generator that performs iteration. **Verify:** Partially consume the repaired file generator and then finish it; assert rows remain readable until exhaustion and the handle closes afterward.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a function that returns a generator expression over a file after the surrounding `with` block has already closed the file. Own the `with` block inside the generator that...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Add a quarantine side channel for malformed rows without making the clean stream eagerly load the entire file. **Progressive hint:** Pass a callback/list for bounded error records or yield tagged results. **Verify:** Feed valid/invalid/valid rows and assert the clean stream remains lazy, output order is preserved, and accepted plus quarantined counts equal input.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Add a quarantine side channel for malformed rows without making the clean stream eagerly load the entire file. Pass a callback/list for bounded error records or yield tagged res...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
